## Pesquisa Profunda

Um dos casos clássicos de uso Agentic entre áreas de negócios! Isso é enorme.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Implicações comerciais</h2>
            <span style="color:#00bfff;">Um agente de Pesquisa Profunda é amplamente aplicável a qualquer área de negócios e também às suas atividades do dia a dia. Você mesmo pode aproveitar isso!
            </span>
        </td>
    </tr>
</table>

In [1]:
from agents import Agent, WebSearchTool, trace, Runner, gen_trace_id, function_tool
from agents.model_settings import ModelSettings
from pydantic import BaseModel, Field
from dotenv import load_dotenv
import asyncio
import sendgrid
import os
from sendgrid.helpers.mail import Mail, Email, To, Content
from typing import Dict
from IPython.display import display, Markdown

In [2]:
load_dotenv(override=True)

True

## Ferramentas hospedadas da OpenAI

O OpenAI Agents SDK inclui as seguintes ferramentas hospedadas:

A `WebSearchTool` permite que um agente pesquise na web.  
A `FileSearchTool` permite recuperar informações dos seus OpenAI Vector Stores.  
A `ComputerTool` permite automatizar tarefas de uso do computador, como tirar capturas de tela e clicar.

### Nota importante — custo da API do WebSearchTool

Isso me custa 2,5 centavos por chamada para a OpenAI WebSearchTool. Isso pode somar US$ 2–3 nos próximos 2 labs. Usaremos ferramentas de busca gratuitas e de baixo custo em outras plataformas, então sinta-se à vontade para pular a execução disso se o custo for uma preocupação. Além disso, o aluno Christian W. apontou que a OpenAI às vezes pode cobrar por várias buscas em uma única chamada, portanto às vezes pode custar mais de 2,5 centavos por chamada.

Os custos estão aqui: https://platform.openai.com/docs/pricing#web-search

In [ ]:
INSTRUCTIONS = "Você é um assistente de pesquisa. Dado um termo de busca, você pesquisa na web por esse termo e \
produz um resumo conciso dos resultados. O resumo deve ter 2–3 parágrafos e menos de 300 \
palavras. Capture os pontos principais. Escreva de forma sucinta; não há necessidade de frases completas ou boa \
gramática. Isso será consumido por alguém que está sintetizando um relatório, então é vital que você capture a \
essência e ignore qualquer enfeite. Não inclua nenhum comentário adicional além do próprio resumo."

search_agent = Agent(
    name="Agente de Pesquisa",
    instructions=INSTRUCTIONS,
    tools=[WebSearchTool(search_context_size="low")],
    model="gpt-5-mini",
    model_settings=ModelSettings(tool_choice="required"),
)

In [ ]:
message = "Frameworks mais recentes de Agentes de IA em 2025"

with trace("Pesquisa"):
    result = await Runner.run(search_agent, message)

display(Markdown(result.final_output))

### Como sempre, dê uma olhada no trace

https://platform.openai.com/traces

### Agora usaremos Structured Outputs e incluiremos uma descrição dos campos

In [ ]:
# See note above about cost of WebSearchTool

HOW_MANY_SEARCHES = 3

INSTRUCTIONS = f"Você é um assistente de pesquisa prestativo. Dada uma pergunta, elabore um conjunto de buscas na web \
a serem realizadas para responder da melhor forma à pergunta. Emita {HOW_MANY_SEARCHES} termos para consultar."

# Use Pydantic para definir o schema da nossa resposta — isso é conhecido como "Structured Outputs"
# Agradecimentos ao aluno Wes C. por descobrir e corrigir um bug complicado nisso!

class WebSearchItem(BaseModel):
    reason: str = Field(description="Seu raciocínio sobre por que essa busca é importante para a pergunta.")

    query: str = Field(description="O termo de busca a ser usado na pesquisa web.")


class WebSearchPlan(BaseModel):
    searches: list[WebSearchItem] = Field(description="Uma lista de buscas na web a serem realizadas para melhor responder à pergunta.")


planner_agent = Agent(
    name="PlannerAgent",
    instructions=INSTRUCTIONS,
    model="gpt-5-mini",
    output_type=WebSearchPlan,
)

In [ ]:

message = "Frameworks mais recentes de Agentes de IA em 2025"

with trace("Pesquisa"):
    result = await Runner.run(planner_agent, message)
    print(result.final_output)

In [ ]:
@function_tool
def send_email(subject: str, html_body: str) -> Dict[str, str]:
    """ Envie um e-mail com o assunto e o corpo HTML fornecidos """
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email = Email("contato@santsoft.com") # Altere isto para o seu e-mail verificado
    to_email = To("santclear@gmail.com") # Altere isto para o seu e-mail
    content = Content("text/html", html_body)
    mail = Mail(from_email, to_email, subject, content).get()
    response = sg.client.mail.send.post(request_body=mail)
    return {"status": "success"}

In [ ]:
send_email

In [ ]:
INSTRUCTIONS = """Você é capaz de enviar um e-mail em HTML bem formatado com base em um relatório detalhado.
Você receberá um relatório detalhado. Você deve usar sua ferramenta para enviar um e-mail, fornecendo o 
relatório convertido em HTML limpo e bem apresentado, com uma linha de assunto apropriada."""

email_agent = Agent(
    name="Agente de E-mail",
    instructions=INSTRUCTIONS,
    tools=[send_email],
    model="gpt-5-mini",
)



In [ ]:
INSTRUCTIONS = (
    "Você é um pesquisador sênior encarregado de escrever um relatório coeso para uma consulta de pesquisa. "
    "Você receberá a consulta original e algumas pesquisas iniciais feitas por um assistente de pesquisa.\n"
    "Primeiro, você deve elaborar um esboço do relatório que descreva a estrutura e o "
    "fluxo do relatório. Em seguida, gere o relatório e retorne-o como sua saída final.\n"
    "A saída final deve estar em formato markdown e deve ser longa e detalhada. Mire "
    "em 5-10 páginas de conteúdo, pelo menos 1000 palavras."
)


class ReportData(BaseModel):
    short_summary: str = Field(description="Um breve resumo de 2–3 frases das conclusões.")

    markdown_report: str = Field(description="O relatório final")

    follow_up_questions: list[str] = Field(description="Tópicos sugeridos para pesquisar mais a fundo")


writer_agent = Agent(
    name="Agente Redator",
    instructions=INSTRUCTIONS,
    model="gpt-5-mini",
    output_type=ReportData,
)

### As próximas 3 funções vão planejar e executar a busca usando planner_agent e search_agent

In [39]:
async def plan_searches(query: str):
    """ Use o planner_agent para planejar quais buscas executar para a consulta """
    print("Planejando pesquisas...")
    result = await Runner.run(planner_agent, f"Query: {query}")
    print(f"Realizará {len(result.final_output.searches)} pesquisas")
    return result.final_output

async def perform_searches(search_plan: WebSearchPlan):
    """ Chame search() para cada item no plano de busca """
    print("Pesquisando...")
    tasks = [asyncio.create_task(search(item)) for item in search_plan.searches]
    results = await asyncio.gather(*tasks)
    print("Pesquisa concluída")
    return results

async def search(item: WebSearchItem):
    """ Use o agente de busca para executar uma pesquisa na web para cada item do plano de busca """
    input = f"Termo de busca: {item.query}\nMotivo da busca: {item.reason}"
    result = await Runner.run(search_agent, input)
    return result.final_output

### As próximas 2 funções escrevem um relatório e o enviam por e-mail

In [40]:
async def write_report(query: str, search_results: list[str]):
    """ Use o agente redator para escrever um relatório com base nos resultados da pesquisa"""
    print("Pensando sobre o relatório...")
    input = f"Original query: {query}\nSummarized search results: {search_results}"
    result = await Runner.run(writer_agent, input)
    print("Relatório concluído")
    return result.final_output

async def send_email(report: ReportData):
    """ Use o agente de e-mail para enviar um e-mail com o relatório """
    print("Escrevendo e-mail...")
    result = await Runner.run(email_agent, report.markdown_report)
    print("E-mail enviado")
    return report

### Hora do show!

In [ ]:
query ="Frameworks mais recentes de Agentes de IA em 2025"

with trace("Rastro da pesquisa"):
    print("Iniciando pesquisa...")
    search_plan = await plan_searches(query)
    search_results = await perform_searches(search_plan)
    report = await write_report(query, search_results)
    await send_email(report)  
    print("Viva!")




### Como sempre, dê uma olhada no trace

https://platform.openai.com/traces

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thanks.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00cc00;">Parabéns pelo seu progresso, e um pedido</h2>
            <span style="color:#00cc00;">Você chegou a um momento importante do curso; você criou um agente valioso usando um dos frameworks de agentes mais recentes. Você elevou suas habilidades e desbloqueou novas possibilidades comerciais. Tire um momento para celebrar seu sucesso!<br/><br/>Algo que devo pedir — meu editor me daria uma bronca se eu não mencionasse isso. Se você puder avaliar o curso na Udemy, eu ficaria muito grato: é a maneira mais importante pela qual a Udemy decide se mostra o curso para outras pessoas e isso faz uma enorme diferença.<br/><br/>E outro lembrete para <a href="https://www.linkedin.com/in/eddonner/">conectar-se comigo no LinkedIn</a> se desejar! Se quiser postar sobre seu progresso no curso, por favor me marque e eu participarei para aumentar sua exposição.
            </span>
        </td>
    </tr>